# Install pyspark

In [ ]:
!pip install pyspark

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.8/310.8 MB 3.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pyspark: filename=pyspark-3.4.0-py2.py3-none-any.whl size=311317130 sha256=25874ac4f5d13b9444cd7f6db29694c0c17678a18750035e714ee6b596352b60
  Stored in directory: /root/.cache/pip/wheels/7b/1b/4b/3363a1d04368e7ff0d408e57ff57966fcdf00583774e761327
Successfully built pyspark


In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").getOrCreate()
sc = spark.sparkContext

In [ ]:
# Montage drive (optionnel)
from google.colab import drive
drive.mount('/My_drive/')

Mounted at /My_drive/


# TP numero 1

In [ ]:
import pyspark.sql.functions as F
import pyspark.sql.types as T
sc

<SparkContext master=local[*] appName=pyspark-shell>

In [ ]:
#Charger un csv (modifier le path)
path="/My_drive/My Drive/Cours prof Big data/2022 GIS5/Projet/offensive_comment_small.csv"
df = spark.read.csv(path,header=True, sep=";")
print(df.count())
df.show()

10000
+------------+--------------------+-----+------------+-------+------+------+-------------+
|          id|        comment_text|toxic|severe_toxic|obscene|threat|insult|identity_hate|
+------------+--------------------+-----+------------+-------+------+------+-------------+
| 86236166566|How can Ceha use ...|    0|           0|      0|     0|     0|            0|
| 67516955649|Blah... Childlove...|    0|           0|      0|     0|     0|            0|
| 22144619496|WTH DUDE Deleting...|    0|           0|      0|     0|     1|            0|
| 10428262239|"'""I''ve added a...|    0|           0|      0|     0|     0|            0|
| 82779776033|Violation Your re...|    0|           0|      0|     0|     0|            0|
| 72769334443|take care  16:55,...|    0|           0|      0|     0|     0|            0|
| 80250530594|'The second examp...|    0|           0|      0|     0|     0|            0|
| 40748094103|"'"" Bye I''m not...|    1|           0|      1|     0|     1|        

# Nettoyer la chaine de caractère

In [ ]:
# Dans un premier temps, faire une fonction qui fonctionne sur un exemple simple
# Doc : https://spacy.io/usage/linguistic-features#pos-tagging
import spacy

nlp = spacy.load("en_core_web_sm", disable=["ner"])

def clean_text(text):
  tab_result = []
  # Ne garder que les LEMME, enlever les STOPWORDS et les mots de moins de 2 charactères
  # A vous de coder
  return tab_result

# modifier la phrase
clean_text("This IS a test, we are currently in a big data lesson and we try spacy")

In [ ]:
# A l'aide du premier TP et des UDF, appliquez votre fonction sur la colonne "comment_text"
df_clean_text= df.withColumn(
    '', # ====> preciser le nom de votre colonnes (words)
    F.udf(

        , # ===> Preciser le type de sortie, sachant que c'est une liste de chaine de charactères
    )("") # ===> Preciser la/les colonnes à utiliser
)

# Correction Spacy


In [ ]:
# Correction

import spacy

nlp = spacy.load("en_core_web_sm", disable=["ner"])

def clean_text(text):
  tab_result = []
  # Ne garder que les LEMME, enlever les STOPWORDS et les mots de moins de 2 charactères
  doc = nlp(text)
  for token in doc :
    if token.is_stop == False and len(token.text)>2:
      tab_result.append(token.lemma_)
  # A vous de coder
  return tab_result

# modifier la phrase
clean_text("This IS a test, we are currently in a big data lesson and we try spacy")

['test', 'currently', 'big', 'datum', 'lesson', 'try', 'spacy']

In [ ]:
# A l'aide du premier TP et des UDF, appliquez votre fonction sur la colonne "comment_text"
df_clean_text= df.withColumn(
    'words', # ====> preciser le nom de votre colonnes (words)
    F.udf(
        #lambda a : clean_text(a) # ===> Preciser clean_text
        clean_text
        ,T.ArrayType(T.StringType()) # ===> Preciser le type de sortie, sachant que c'est une liste de chaine de charactères
    )("comment_text") # ===> Preciser la/les colonnes à utiliser
)

df_clean_text.show()

+------------+--------------------+-----+------------+-------+------+------+-------------+--------------------+
|          id|        comment_text|toxic|severe_toxic|obscene|threat|insult|identity_hate|               words|
+------------+--------------------+-----+------------+-------+------+------+-------------+--------------------+
| 86236166566|How can Ceha use ...|    0|           0|      0|     0|     0|            0|[Ceha, use, word,...|
| 67516955649|Blah... Childlove...|    0|           0|      0|     0|     0|            0|[Blah, ..., Child...|
| 22144619496|WTH DUDE Deleting...|    0|           0|      0|     0|     1|            0|[wth, dude, delet...|
| 10428262239|"'""I''ve added a...|    0|           0|      0|     0|     0|            0|[I''ve, add, cita...|
| 82779776033|Violation Your re...|    0|           0|      0|     0|     0|            0|[violation, recen...|
| 72769334443|take care  16:55,...|    0|           0|      0|     0|     0|            0|[care, 16:55, 

## TRES IMPORTANT : Sauvegarder dans un fichier parquet, recharger vos données

In [ ]:
df_clean_txt.write.parquet("/My_drive/My Drive/Cours prof Big data/2022 GIS5/Projet/first_clean.pqt", mode="overwrite")

In [ ]:
df_clean_txt = spark.read.parquet("/My_drive/My Drive/Cours prof Big data/2022 GIS5/Projet/first_clean.pqt")

Prochaine étapes :
- Fit, transform sur un algo de "vectorisation" CountVectorizer, TFIDF, W2V etc...
- Entrainer un modele sur la sortie de la "vectorisation"

Tips :
- Attention aux types, valeurs manquantes etc...
- Evaluer votre modèle à chaque tests
- Tester différentes approches !
- cf slides

In [ ]:
from pyspark.ml.feature import CountVectorizer
# fit a CountVectorizerModel from the corpus.
cv = CountVectorizer()

model = cv.fit(df_clean_txt)

result = model.transform(df_clean_txt)
result.show(1,truncate=False)


+-----------+--------------------------------------------------------------------------------------------------------------+-----+------------+-------+------+------+-------------+---------------------------------------------------------+----------+
|id         |comment_text                                                                                                  |toxic|severe_toxic|obscene|threat|insult|identity_hate|words                                                    |features  |
+-----------+--------------------------------------------------------------------------------------------------------------+-----+------------+-------+------+------+-------------+---------------------------------------------------------+----------+
|86236166566|How can Ceha use words POV in every other sentence and not get banned, yet I may not?Here is the map btw.(  ).|0    |0           |0      |0     |0     |0            |[Ceha, use, word, POV, sentence, ban, not?here, map, btw]|(13,[],[])|
+---

In [ ]:
model.vocabulary[127]

'ban'

In [ ]:
from pyspark.ml.classification import RandomForestClassifier
# Vous pouvez egalement utiliser d'autre algorithme
# labelCol=toxic <== Pensez à la caster

Pour la suite :
Vous êtes en autonomie, à vous d'experimenter différentes approches.
Chaque approche (même si elle ne donne pas de bon résultats) est intéressante à documenter.

Quelques pistes (Slides 63+64 du cours) :
- Mesurer les performances de vos modèles (et justifier le choix de votre score)
- Tenir un excel/autre doc à jour avec vos scores pour chaque experimentation
- Gerer les données déséquilibrer
- Tester un Word2Vec
- Ajouter un IDF après votre countVectorizer